## Cell 1 — Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Cell 2 — Load saved predictions and labels

In [ ]:
import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, confusion_matrix, recall_score
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)

DRIVE_DIR   = "/content/drive/MyDrive/Corneal_v2"
RESULTS_DIR = "/content/drive/MyDrive/CNN_results_v2"
CSV_FILE = os.path.join(DRIVE_DIR, "Final_Features_MultiLabel_updated.csv")
SEV_FILE = os.path.join(DRIVE_DIR, "Severity_Labels_PerDisease_updated.xlsx")
SEV_DIR  = os.path.join(RESULTS_DIR, "severity_densenet121")           # Severity_DenseNet121_v2
OUT_DIR  = os.path.join(RESULTS_DIR, "combined_final")
os.makedirs(OUT_DIR, exist_ok=True)

# disease predictions: full comparison notebook (preferred) or the fusion-only notebook
DIS_CANDIDATES = [os.path.join(RESULTS_DIR, "final_densenet121", "test_predictions_all.csv"),          # Fusion_DenseNet121_Retrain_v2
                  os.path.join(RESULTS_DIR, "final_fusion_densenet121", "test_predictions_fusion.csv")]  # fusion-only, BACKBONE=densenet121
DIS_FILE = next((f for f in DIS_CANDIDATES if os.path.exists(f)), None)
assert DIS_FILE, "Disease predictions not found - run Fusion_DenseNet121_Retrain_v2 to the end first"
for f in (CSV_FILE, SEV_FILE, os.path.join(SEV_DIR, "summary_severity.json")):
    assert os.path.exists(f), f"Missing: {f}"

DIS_COLS = ["FECD", "Guttata", "Folds", "PolyPleo", "LowCD"]

dis      = pd.read_csv(DIS_FILE)
sev_pred = pd.read_csv(os.path.join(SEV_DIR, "test_predictions_severity.csv"))
sev_oof  = pd.read_csv(os.path.join(SEV_DIR, "oof_predictions.csv"))
sev_sum  = json.load(open(os.path.join(SEV_DIR, "summary_severity.json")))
labels   = pd.read_excel(SEV_FILE, sheet_name="Severity_Labels", keep_default_na=False, na_values=[""])
feat     = pd.read_csv(CSV_FILE)

SEV_HEADS = sev_sum["severity_heads"]            # e.g. {"FECDgroup": [...], "Folds": [...], "Guttata": [...]}
SEV_THR   = sev_sum["severity_thresholds"]
HEAD_NAME = {"FECDgroup": "FECD group", "Folds": "Folds", "Guttata": "Guttata"}
HAS_NUM   = all(f"P_Num_{c}" in dis.columns for c in DIS_COLS)

print("Disease predictions :", DIS_FILE, dis.shape)
print("Severity predictions:", sev_pred.shape)
print("Severity outputs    :", list(SEV_HEADS))
print("Severity thresholds :", {k: round(v, 3) for k, v in SEV_THR.items()})

Disease predictions : /content/drive/MyDrive/CNN_results_v2/final_densenet121/test_predictions_all.csv (305, 38)
Severity predictions: (305, 21)
Severity outputs    : ['FECDgroup', 'Folds', 'Guttata']
Severity thresholds : {'FECDgroup': 0.57, 'Folds': 0.491, 'Guttata': 0.434}


## Cell 3 — Combine into one table (one row per test cornea)

In [ ]:
t = (dis[["Tissue_ID", "Combination"] + DIS_COLS + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]]
     .merge(sev_pred[["Tissue_ID", "Label_Origin"] + [f"PSev_{h}" for h in SEV_HEADS]], on="Tissue_ID")
     .merge(labels[["Tissue_ID"] + [f"{h}_Sev" for h in SEV_HEADS] + [c for c in ["Guttata_Sev", "Overall_Sev"]
                                                                       if c not in [f"{h}_Sev" for h in SEV_HEADS]]], on="Tissue_ID")
     .merge(feat[["Tissue_ID", "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"]], on="Tissue_ID", how="left"))
print(f"Test corneas combined: {len(t)} | disease file {len(dis)} | severity file {len(sev_pred)}")
assert len(t) == len(dis) == len(sev_pred), "The two notebooks used different test rows - re-run both with the same CSV/images"
t.head(3)

Test corneas combined: 305 | disease file 305 | severity file 305


,Tissue_ID,Combination,FECD,Guttata,Folds,PolyPleo,LowCD,P_Fus_FECD,P_Fus_Guttata,P_Fus_Folds,P_Fus_PolyPleo,P_Fus_LowCD,Pred_Fus_FECD,Pred_Fus_Guttata,Pred_Fus_Folds,Pred_Fus_PolyPleo,Pred_Fus_LowCD,Label_Origin,PSev_FECDgroup,PSev_Folds,PSev_Guttata,FECDgroup_Sev,Folds_Sev,Guttata_Sev,Overall_Sev,ECD_CD_cells_per_mm2,CV,HEX_percent,Donor_Age
0,NEB-2020-09-12663 R,Normal,0,0,0,0,0,0.003929,0.318359,0.871094,0.189453,0.000207,0,0,1,0,0,Original,0.899414,0.740234,0.810547,None,None,None,None,2740.0,41.0,56.0,72.0
1,NEB-2020-09-12667 R,Normal,0,0,0,0,0,0.002728,0.142334,0.827637,0.119385,0.000083,0,0,1,0,0,Original,0.537109,0.284180,0.508301,None,None,None,None,2833.0,34.0,61.0,69.0
2,NEB-2020-09-12673 L,FECD + PolyPleo + LowCD,1,0,0,1,1,0.604492,0.056213,0.782715,0.787109,0.917480,1,0,1,1,1,Original,0.547852,0.732422,0.688477,Mild,None,None,Mild,2016.0,35.0,48.0,59.0


## Cell 4 — Severity cut-off options (training CV)
For each severity output: how many Mod/Sev corneas are caught vs how many Mild ones are wrongly flagged at different cut-offs.
Use this to choose the cut-offs in Cell 5.

In [ ]:
for h in SEV_HEADS:
    tr = sev_oof.merge(labels[["Tissue_ID", f"{h}_Sev"]], on="Tissue_ID")
    tr = tr[tr[f"{h}_Sev"].isin(["Mild", "Moderate", "Severe"])]
    y = tr[f"{h}_Sev"].isin(["Moderate", "Severe"]).astype(int).values
    p = tr[f"PSev_{h}"].values
    n_mild, n_mod = int((y == 0).sum()), int(y.sum())
    rows = []
    for thr in [SEV_THR[h], 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:
        yhat = (p >= thr).astype(int)
        rows.append({"Cut-off": round(thr, 3), "Sensitivity (catch Mod/Sev)": recall_score(y, yhat, zero_division=0),
                     "Specificity (keep Mild as Mild)": recall_score(y, yhat, pos_label=0, zero_division=0),
                     f"Mild wrongly flagged (of {n_mild})": int(((yhat == 1) & (y == 0)).sum()),
                     f"Mod/Sev missed (of {n_mod})": int(((yhat == 0) & (y == 1)).sum())})
    print(f"\n{HEAD_NAME.get(h, h)} severity cut-off options (training CV, {n_mild} Mild / {n_mod} Mod+Sev) - first row = automatic (Youden):")
    display(pd.DataFrame(rows).round(3))


FECD group severity cut-off options (training CV, 201 Mild / 185 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 201),Mod/Sev missed (of 185)
0,0.57,0.649,0.801,40,65
1,0.30,0.892,0.438,113,20
2,0.35,0.859,0.493,102,26
3,0.40,0.805,0.557,89,36
4,0.45,0.773,0.647,71,42
5,0.50,0.708,0.701,60,54
6,0.55,0.654,0.771,46,64
7,0.60,0.589,0.811,38,76



Folds severity cut-off options (training CV, 390 Mild / 225 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 390),Mod/Sev missed (of 225)
0,0.491,0.711,0.769,90,65
1,0.300,0.880,0.513,190,27
2,0.350,0.840,0.559,172,36
3,0.400,0.782,0.656,134,49
4,0.450,0.747,0.731,105,57
5,0.500,0.680,0.782,85,72
6,0.550,0.613,0.826,68,87
7,0.600,0.573,0.874,49,96



Guttata severity cut-off options (training CV, 112 Mild / 40 Mod+Sev) - first row = automatic (Youden):


,Cut-off,Sensitivity (catch Mod/Sev),Specificity (keep Mild as Mild),Mild wrongly flagged (of 112),Mod/Sev missed (of 40)
0,0.434,0.700,0.705,33,12
1,0.300,0.800,0.536,52,8
2,0.350,0.775,0.607,44,9
3,0.400,0.725,0.679,36,11
4,0.450,0.625,0.705,33,15
5,0.500,0.550,0.750,28,18
6,0.550,0.425,0.812,21,23
7,0.600,0.350,0.893,12,26


## Cell 5 — Choose cut-offs

In [ ]:
THR = dict(SEV_THR)              # automatic cut-offs from Severity_DenseNet121_v2
# THR["Folds"] = 0.40            # <- uncomment / edit after looking at the tables above
FECD_SEV_METHOD = "model"        # "model" = severity notebook output | "cd_rule" = cell density < 1600 -> Mod/Sev
print({k: round(v, 3) for k, v in THR.items()}, "| FECD method:", FECD_SEV_METHOD)

{'FECDgroup': 0.57, 'Folds': 0.491, 'Guttata': 0.434} | FECD method: model


## Cell 6 — Final report per cornea + overall severity confusion matrix

In [ ]:
ORD = {"None": 0, "Mild": 1, "Mod/Sev": 2}

def head_severity(r, h):
    if h == "FECDgroup" and FECD_SEV_METHOD == "cd_rule":
        return "Mod/Sev" if r["ECD_CD_cells_per_mm2"] < 1600 else "Mild"
    return "Mod/Sev" if r[f"PSev_{h}"] >= THR[h] else "Mild"

def cornea_report(r):
    found = [c for c in DIS_COLS if r[f"Pred_Fus_{c}"] == 1]          # diseases from the fusion notebook
    sevs = [(HEAD_NAME.get(h, h), head_severity(r, h)) for h, ds in SEV_HEADS.items() if any(c in found for c in ds)]
    if "Guttata" in found and not any("Guttata" in ds for ds in SEV_HEADS.values()):
        sevs.append(("Guttata", "Mild"))                               # only if no Guttata severity output
    overall = max([s for _, s in sevs], key=lambda s: ORD[s]) if sevs else "None"
    return pd.Series({"Predicted diseases": " + ".join(found) if found else "Normal",
                      "Predicted severity": ", ".join(f"{d}: {s}" for d, s in sevs) if sevs else "-",
                      "Predicted overall": overall})

to3 = lambda s: s.astype(str).replace({"Moderate": "Mod/Sev", "Severe": "Mod/Sev", "?": "Mod/Sev", "nan": "None"})
rep = pd.concat([t, t.apply(cornea_report, axis=1)], axis=1)
rep["True overall"] = to3(rep["Overall_Sev"])

L = ["None", "Mild", "Mod/Sev"]
cm = pd.DataFrame(confusion_matrix(rep["True overall"], rep["Predicted overall"], labels=L),
                  index=[f"True {l}" for l in L], columns=[f"Pred {l}" for l in L])
print("FINAL end-to-end overall severity (test): fusion diseases + severity model")
display(cm)
agree = (rep["True overall"] == rep["Predicted overall"]).mean()
sev_cm = pd.DataFrame(sev_sum.get("overall_confusion", {}))
sev_alone = np.trace(sev_cm.values) / sev_cm.values.sum() if sev_cm.size else np.nan
print(f"Exact agreement: {agree:.1%}   (severity notebook alone: {sev_alone:.1%})")
print(f"Mod/Sev corneas called Normal: {int(cm.loc['True Mod/Sev', 'Pred None'])} of {int(cm.loc['True Mod/Sev'].sum())}")

FINAL end-to-end overall severity (test): fusion diseases + severity model


,Pred None,Pred Mild,Pred Mod/Sev
True None,53,41,7
True Mild,10,79,44
True Mod/Sev,0,20,51


Exact agreement: 60.0%   (severity notebook alone: 65.9%)
Mod/Sev corneas called Normal: 0 of 71


## Cell 7 — Severity per disease on the DETECTED cases (test)

In [ ]:
rows = []
for h, ds in SEV_HEADS.items():
    true_col = f"{h}_Sev"
    m = rep[true_col].isin(["Mild", "Moderate", "Severe"]) & rep[[f"Pred_Fus_{c}" for c in ds]].max(axis=1).eq(1)
    yt = rep.loc[m, true_col].isin(["Moderate", "Severe"]).astype(int)
    yp = rep.loc[m].apply(lambda r: head_severity(r, h), axis=1).eq("Mod/Sev").astype(int) if m.any() else pd.Series(dtype=int)
    rows.append({"Disease": HEAD_NAME.get(h, h), "Truly present": int(rep[true_col].isin(["Mild", "Moderate", "Severe"]).sum()),
                 "Detected": int(m.sum()), "Mild / Mod+Sev (detected)": f"{int((yt == 0).sum())} / {int(yt.sum())}",
                 "Severity correct": f"{(yt == yp).mean():.1%}" if len(yt) else "-",
                 "Sensitivity": recall_score(yt, yp, zero_division=0) if len(yt) else np.nan,
                 "Specificity": recall_score(yt, yp, pos_label=0, zero_division=0) if len(yt) else np.nan})
per_dis = pd.DataFrame(rows).set_index("Disease")
per_dis.to_csv(os.path.join(OUT_DIR, "severity_on_detected_cases.csv"))
print("Severity per disease on DETECTED cases (test)"); display(per_dis.round(3))

Severity per disease on DETECTED cases (test)


,Truly present,Detected,Mild / Mod+Sev (detected),Severity correct,Sensitivity,Specificity
Disease,,,,,,
FECD group,98,83,43 / 40,78.3%,0.775,0.791
Folds,153,131,78 / 53,76.3%,0.774,0.756
Guttata,37,28,19 / 9,67.9%,0.444,0.789


## Cell 8 — Example reports and under-called Mod/Sev corneas

In [ ]:
show = ["Tissue_ID", "Combination", "Predicted diseases", "Predicted severity", "Predicted overall", "True overall"]
print("Example final cornea reports (test):")
display(rep[show].sample(min(15, len(rep)), random_state=3).reset_index(drop=True))
print("\nMistakes where a Mod/Sev cornea was under-called:")
display(rep.loc[(rep["True overall"] == "Mod/Sev") & (rep["Predicted overall"] != "Mod/Sev"), show])

Example final cornea reports (test):


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
0,NEB-2024-08-16268 R,Folds,Folds,Folds: Mild,Mild,Mild
1,NEB-2020-02-12074 L,PolyPleo,Folds + PolyPleo,"FECD group: Mild, Folds: Mod/Sev",Mod/Sev,Mod/Sev
2,NEB-2020-10-12719 R,Folds,Guttata + Folds,"Folds: Mild, Guttata: Mod/Sev",Mod/Sev,Mild
3,NEB-2023-07-15254 R,Normal,Normal,-,None,None
4,NEB-2023-03-14873 R,Normal,Normal,-,None,None
5,NEB-2020-10-12743 L,Normal,Folds,Folds: Mild,Mild,None
6,NEB-2023-02-14732 R,Normal,Normal,-,None,None
7,NEB-2022-11-14454 L,Folds + LowCD,FECD + Guttata + Folds + PolyPleo + LowCD,"FECD group: Mod/Sev, Folds: Mod/Sev, Guttata: ...",Mod/Sev,Mod/Sev
8,NEB-2025-11-17642 L,Normal,Normal,-,None,None
9,NEB-2023-02-14717 R,Folds,Folds,Folds: Mild,Mild,Mild



Mistakes where a Mod/Sev cornea was under-called:


,Tissue_ID,Combination,Predicted diseases,Predicted severity,Predicted overall,True overall
72,NEB-2023-04-14936 R,Folds,Folds,Folds: Mild,Mild,Mod/Sev
87,NEB-2023-05-15068 R,Folds,Folds,Folds: Mild,Mild,Mod/Sev
88,NEB-2023-05-15084 R,Guttata + Folds,Guttata + Folds,"Folds: Mild, Guttata: Mild",Mild,Mod/Sev
158,NEB-2024-08-16275 L,Guttata + Folds,Guttata,Guttata: Mild,Mild,Mod/Sev
192,NEB-2025-10-17595 R,Guttata,Guttata,Guttata: Mild,Mild,Mod/Sev
196,NEB-2025-11-17619 L,FECD + PolyPleo,FECD + Folds + PolyPleo + LowCD,"FECD group: Mild, Folds: Mild",Mild,Mod/Sev
210,NEB-2020-06-12374 L,FECD + Guttata + PolyPleo + LowCD,FECD + Guttata + PolyPleo + LowCD,"FECD group: Mild, Guttata: Mild",Mild,Mod/Sev
240,NEB-2024-09-16332 L,PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mod/Sev
248,NEB-2025-08-17405 R,Guttata + Folds + PolyPleo + LowCD,FECD + PolyPleo + LowCD,FECD group: Mild,Mild,Mod/Sev
258,NEB-2020-01-11896 R,Folds + PolyPleo,Folds,Folds: Mild,Mild,Mod/Sev


## Cell 9 — Save

In [ ]:
keep = (["Tissue_ID", "Label_Origin", "Combination"] + DIS_COLS
        + [c for c in [f"{h}_Sev" for h in SEV_HEADS] + ["Guttata_Sev"] if c in rep.columns][:len(SEV_HEADS) + 1]
        + ["True overall"] + [f"P_Fus_{c}" for c in DIS_COLS] + [f"Pred_Fus_{c}" for c in DIS_COLS]
        + [f"PSev_{h}" for h in SEV_HEADS]
        + ["Predicted diseases", "Predicted severity", "Predicted overall", "ECD_CD_cells_per_mm2", "CV", "HEX_percent", "Donor_Age"])
keep = list(dict.fromkeys(keep))
rep[keep].to_csv(os.path.join(OUT_DIR, "Final_Cornea_Reports_test.csv"), index=False)
cm.to_csv(os.path.join(OUT_DIR, "overall_severity_confusion.csv"))
json.dump({"disease_model": f"DenseNet121 image+numbers fusion ({os.path.basename(DIS_FILE)})",
           "severity_model": "DenseNet121 multi-task (Severity_DenseNet121_v2)", "n_test": len(rep),
           "severity_thresholds": THR, "fecd_sev_method": FECD_SEV_METHOD,
           "overall_exact_agreement": round(float(agree), 4)},
          open(os.path.join(OUT_DIR, "combined_summary.json"), "w"), indent=2)
print("Saved to", OUT_DIR); print(os.listdir(OUT_DIR))

Saved to /content/drive/MyDrive/CNN_results_v2/combined_final
['severity_on_detected_cases.csv', 'Final_Cornea_Reports_test.csv', 'overall_severity_confusion.csv', 'combined_summary.json']


## Cell 10 — Disease AUC split by who made the label (original vs model-assisted 2023)

In [ ]:
g = dis.merge(sev_pred[["Tissue_ID", "Label_Origin"]], on="Tissue_ID")
def auc_or_nan(y, p):
    return roc_auc_score(y, p) if 0 < y.sum() < len(y) else np.nan
rows = []
for org in ["All test", "Original", "Model-assisted (2023)"]:
    s = g if org == "All test" else g[g["Label_Origin"] == org]
    for c in DIS_COLS:
        r = {"Group": f"{org} (n={len(s)})", "Disease": c, "Cases": int(s[c].sum()),
             "Image + numbers AUC": auc_or_nan(s[c].values, s[f"P_Fus_{c}"].values)}
        if HAS_NUM: r["Numbers only AUC"] = auc_or_nan(s[c].values, s[f"P_Num_{c}"].values)
        rows.append(r)
res = pd.DataFrame(rows)
print("Disease AUC on the SAME test predictions, split by who made the label")
display(res.pivot(index="Disease", columns="Group", values="Image + numbers AUC").round(3))
if HAS_NUM:
    print("\nNumbers-only model, same split (for comparison)")
    display(res.pivot(index="Disease", columns="Group", values="Numbers only AUC").round(3))
print("\nNumber of diseased corneas per group")
display(res.pivot(index="Disease", columns="Group", values="Cases"))
res.to_csv(os.path.join(OUT_DIR, "disease_auc_by_label_origin.csv"), index=False)

Disease AUC on the SAME test predictions, split by who made the label


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,0.918,0.965,0.872
Folds,0.804,0.843,0.784
Guttata,0.880,0.910,0.872
LowCD,0.985,1.000,0.977
PolyPleo,0.852,0.949,0.801



Numbers-only model, same split (for comparison)


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,0.870,0.940,0.874
Folds,0.611,0.581,0.614
Guttata,0.599,0.688,0.567
LowCD,0.982,1.000,0.975
PolyPleo,0.858,0.877,0.848



Number of diseased corneas per group


Group,All test (n=305),Model-assisted (2023) (n=103),Original (n=202)
Disease,,,
FECD,28,18,10
Folds,154,42,112
Guttata,38,9,29
LowCD,75,24,51
PolyPleo,69,22,47
